# Analysis of pp events: event activity observables

Firstly, we load libraries and enable multi-threading, which allows event processing across multiple CPU cores. We also connect RDataFrame to the TTree in our created events.root file.

In [44]:
import ROOT
import numpy as np

ROOT.EnableImplicitMT() # enable multi-threading (optional, but can speed up processing)

df = ROOT.RDataFrame("events", "events.root") # connect RDataFrame to the TTree in events.root

### Multiplicity distribution

In [45]:
histo_Nch = df.Histo1D(("histo_Nch", "Multiplicity;N_{ch};Normalized Events", 70, 0, 70), "multiplicity") # histogram of the multiplicity distribution
histo_normalized_Nch = histo_Nch.GetValue() # get the histogram object from the RDataFrame
canvas_mult =  ROOT.TCanvas("canvas_mult", "Multiplicity", 800, 600) # create a canvas to draw the histogram
canvas_mult.SetLogy() # set logarithmic scale for y-axis

if histo_Nch.Integral() > 0: # check if the histogram has entries to avoid division by zero
    histo_normalized_Nch.Scale(1.0 / histo_Nch.Integral()) # normalize the histogram
histo_normalized_Nch.SetLineColor(ROOT.kRed + 1) # set line color to slightly darker (+1) red
histo_normalized_Nch.SetFillColorAlpha(ROOT.kRed, 0.1) # set fill color to red with 10% transparency (alpha = 0.1)

histo_normalized_Nch.Draw("HIST") # draw the histogram
canvas_mult.SaveAs("img/multiplicity.pdf") # save the canvas as a PDF file

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_mult
Info in <TCanvas::Print>: pdf file img/multiplicity.pdf has been created


### Unweighted Spherocity $S_0^{p_\mathrm{T} = 1}$ distribution

calculated only for $N_\mathrm{ch}(|\eta| < 1) > 10$

$$
S_0^{p_\mathrm{T} = 1} = \frac{\pi^2}{4} \min_{\hat{n}} \left( \frac{\sum_i |\hat{u}_{\mathrm{T}, i} \times \hat{n}|}{N_\mathrm{trks}} \right)^2
$$

In [46]:
# extract px, py, pT. and multiplicity from the RDataFrame into a dictionary of NumPy arrays -- this loads the data into RAM!!! BE CAREFUL WITH LARGE DATASETS --> better option is to write a function in C++ inside PyROOT as Gemini showed me, but since I'm learning, I will do it this way for now
data = df.AsNumpy(["px", "py", "pT", "multiplicity"])

# define each NumPy arrays separately
pxArr = data["px"]
pyArr = data["py"]
pTArr = data["pT"]
NchArr = data["multiplicity"]

# define the function to calculate the unweighted transverse spherocity for events with multiplicity > 10
def calc_S0pT1(pxEvent, pyEvent, pTEvent, NchEvent):
    # define the unit pT vector
    uxEvent = pxEvent / pTEvent
    uyEvent = pyEvent / pTEvent

    # Since the minimizing axis \hat{n} is always collinear with one of the pT vectors in the event, we can calculate the sum for each pT vector and find the minimum. This is much faster than performing a numerical minimization. 
    absCrossProductMatrix = np.abs(uxEvent[:, np.newaxis]*uyEvent - uyEvent[:, np.newaxis]*uxEvent) # calculate the absolute value of the cross product of each pair of unit pT vectors utilizing [:, np.newaxis], which turns a 1D array into a column vector
    sums = np.sum(absCrossProductMatrix, axis=0) # calculate the sums for each axis (pT vector)
    minSum = np.min(sums) # find the minimum sum

    return np.pi**2 / 4 * (minSum / NchEvent)**2 # calculate and return the unweighted transverse spherocity

# prepare ROOT histogram
canvas_S0pT1 = ROOT.TCanvas("canvas_S0pT1", "S0pT1", 800, 600) # create a canvas to draw the histogram
histo_S0pT1 = ROOT.TH1F("histo_S0pT1", "Unweighted Transverse Spherocity;S_{0}^{pT=1};Normalized Events", 100, 0, 1) # create a histogram for S0pT1 distribution

# event loop
for iEvent in range(len(NchArr)):
    # define observable arrays for each event
    pxEvent = pxArr[iEvent]
    pyEvent = pyArr[iEvent]
    pTEvent = pTArr[iEvent]
    NchEvent = NchArr[iEvent]

    if not NchEvent > 10: # select events with multiplicity > 10
        continue
    else:
        S0pT1 = calc_S0pT1(pxEvent, pyEvent, pTEvent, NchEvent) # calculate the unweighted transverse spherocity for the event
        histo_S0pT1.Fill(S0pT1) # fill the histogram

# customize the histogram and save it
canvas_S0pT1.SetLogy() # set logarithmic scale for y-axis

if histo_S0pT1.Integral() > 0: # check if the histogram is not empty to avoid division by zero
    histo_S0pT1.Scale(1.0 / histo_S0pT1.Integral()) # normalize the histogram
histo_S0pT1.SetLineColor(ROOT.kRed + 1) # set line color to slightly darker (+1) red
histo_S0pT1.SetFillColorAlpha(ROOT.kRed, 0.1) # set fill color to red with 10% transparency (alpha = 0.1)

histo_S0pT1.Draw("HIST") # draw the histogram
canvas_S0pT1.SaveAs("img/S0pT1.pdf") # save the canvas as a PDF file

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_S0pT1
Warning in <TROOT::Append>: Replacing existing TH1: histo_S0pT1 (Potential memory leak).
Info in <TCanvas::Print>: pdf file img/S0pT1.pdf has been created
